In [ ]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.linear_model import LinearRegression


parquet_path = Path("../../../data/raw/NFI/nfi_particle_data_full.parquet")
nfi_df = pd.read_parquet(parquet_path)
nfi_df.shape

(2801667, 106)

In [15]:
# Confirm stub_id + particle_id uniqueness
nfi_df.duplicated(subset=["stub_id", "particle_id"]).sum()

np.int64(0)

In [6]:
# Subset NFI data to only include id + element columns
element_cols = list(nfi_df.loc[:, "ac":"zr"].columns)
elements_df = nfi_df[["stub_id", "particle_id"] + element_cols]
elements_df.columns

Index(['stub_id', 'particle_id', 'ac', 'ag', 'al', 'ar', 'as', 'at', 'au', 'b',
       'ba', 'bi', 'br', 'ca', 'cd', 'ce', 'cl', 'co', 'cr', 'cs', 'cu', 'dy',
       'er', 'eu', 'f', 'fe', 'fr', 'ga', 'gd', 'ge', 'hf', 'hg', 'ho', 'i',
       'in', 'ir', 'k', 'kr', 'la', 'lu', 'mg', 'mn', 'mo', 'n', 'na', 'nb',
       'nd', 'ne', 'ni', 'np', 'o', 'os', 'p', 'pa', 'pb', 'pd', 'pm', 'po',
       'pr', 'pt', 'pu', 'ra', 'rb', 're', 'rh', 'rn', 'ru', 's', 'sb', 'sc',
       'se', 'si', 'sm', 'sn', 'sr', 'ta', 'tb', 'tc', 'te', 'th', 'ti', 'tl',
       'tm', 'u', 'v', 'w', 'xe', 'y', 'yb', 'zn', 'zr'],
      dtype='str')

In [7]:
# Use non-zero rates from particle_eda.ipynb to filter out non-informative elements
nonzero_rates = (elements_df[element_cols] > 0).mean().sort_values(ascending=False)
informative_cols = sorted(nonzero_rates[nonzero_rates > 0.01].index.tolist())
informative_df = elements_df[["stub_id", "particle_id"] + informative_cols]
informative_df.columns

Index(['stub_id', 'particle_id', 'al', 'as', 'ba', 'br', 'ca', 'cl', 'cr',
       'cu', 'fe', 'hg', 'k', 'mg', 'mn', 'mo', 'na', 'ni', 'o', 'p', 'pb',
       's', 'sb', 'si', 'sn', 'sr', 'ti', 'w', 'zn'],
      dtype='str')

In [8]:
# Confirm number of informative features/elements kept
informative_df.shape

(2801667, 29)

In [30]:
# Share of each element as % of grand total sum across all observations
element_sums = informative_df[informative_cols].sum()
grand_total = element_sums.sum()
element_pct = (element_sums / grand_total * 100).sort_values(ascending=False)

print("Top 10 elements by share of total elemental composition (all observations):\n")
display(element_pct.head(10).rename("% of grand total").to_frame().round(2))

Top 10 elements by share of total elemental composition (all observations):



,% of grand total
o,40.31
ba,13.98
fe,8.38
pb,7.69
cu,6.43
s,4.37
sb,4.00
zn,3.26
al,2.63
si,2.36


In [9]:
# Confirm that NFI elemental composition includes oxygen in the percentage (in contrast to NIST)
informative_df["sum_with_oxygen"] = (
    informative_df.loc[:, "al":"zn"].sum(axis=1).round(2)
)
informative_df["sum_without_oxygen"] = (
    informative_df.loc[:, "al":"zn"].drop("o", axis=1).sum(axis=1).round(2)
)
informative_df[["sum_with_oxygen", "sum_without_oxygen"]].head()

,sum_with_oxygen,sum_without_oxygen
0,100.0,77.00
1,100.0,55.35
2,100.0,56.88
3,100.0,61.73
4,100.0,58.39


In [10]:
# Add the GSR / Non-GSR labels from particle_labeled.parquet
p_df = pd.read_parquet("../../../data/processed/particle_labeled.parquet")
p_labeled = p_df[["stub_id", "particle_id", "label"]]
labeled_df = informative_df.copy()
labeled_df = labeled_df.drop(["sum_with_oxygen", "sum_without_oxygen"], axis=1)
labeled_df = labeled_df.merge(p_labeled, on=["stub_id", "particle_id"])
labeled_df.shape

(2801667, 30)

In [11]:
# Confirm Number of GSR vs Non-GSR particles
labeled_df["label"].value_counts()

label
Non_GSR      1216039
GSR          1078946
Ambiguous     506682
Name: count, dtype: int64

In [12]:
# Drop ambiguous particles
bin_labeled_df = labeled_df[labeled_df["label"] != "Ambiguous"]
assert (
    bin_labeled_df.shape[0]
    == labeled_df.shape[0] - labeled_df["label"].value_counts()["Ambiguous"]
)

In [13]:
# New shape
bin_labeled_df.shape

(2294985, 30)

In [ ]:
# Write to parquet
# bin_labeled_df.to_parquet("../../../data/processed/nfi_relevant_elements_labeled.parquet")

In [17]:
# Drop oxygen column and rescale remaining elements to sum to 100 per observation
deoxygenated_df = bin_labeled_df.copy()
deoxygenated_df = deoxygenated_df.drop(columns=["o"])
elem_cols = list(deoxygenated_df.loc[:, "al":"zn"].columns)
row_sums = deoxygenated_df[elem_cols].sum(axis=1)
deoxygenated_df[elem_cols] = deoxygenated_df[elem_cols].div(row_sums, axis=0) * 100
deoxygenated_df.shape

(2294985, 29)

In [18]:
# Verify element columns sum to 100 for each observation
deoxygenated_df[elem_cols].sum(axis=1).round(2).value_counts()

100.0    2294985
Name: count, dtype: int64

In [ ]:
# Write deoxygenated nfi data to parquet
# deoxygenated_df.to_parquet("../../../data/processed/nfi_relevant_elements_deoxygenated.parquet")

## Oxygen Redundancy Analysis

Correlation matrix and R² values for oxygen (`o`) vs. every other informative element.
For simple linear regression, R² = r² (Pearson r squared), so both are derived directly from the correlation matrix.

In [ ]:
# Element columns only
corr_matrix = informative_df[informative_cols].corr()

# Oxygen Pearson r and R² with every other element (R² = r² for simple linear regression)
other_cols = [c for c in informative_cols if c != "o"]
o_corr = corr_matrix.loc["o", other_cols].sort_values(key=abs, ascending=False)
o_r2 = o_corr**2

# Summary table
result = pd.DataFrame({"Pearson r": o_corr.round(4), "R² (simple LR)": o_r2.round(4)})
print("Oxygen correlations with other informative elements (sorted by |r|):\n")
display(result)

# Bar charts: Pearson r and R²
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

colors_r = ["#d62728" if v < 0 else "#1f77b4" for v in o_corr]
o_corr.plot(kind="bar", ax=axes[0], color=colors_r, edgecolor="black", linewidth=0.5)
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].set_title("Oxygen: Pearson r with other informative elements")
axes[0].set_ylabel("Pearson r")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=45)

o_r2.sort_values(ascending=False).plot(
    kind="bar", ax=axes[1], color="steelblue", edgecolor="black", linewidth=0.5
)
axes[1].set_title(
    "Oxygen: R² with other informative elements (simple linear regression)"
)
axes[1].set_ylabel("R²")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=45)

plt.suptitle("Oxygen Relationships with Other Informative Elements", fontsize=13)
plt.tight_layout()
plt.savefig(
    "outputs/NFI_relevant_elements_only__oxygen_pearson_r_with_other_informative_elements.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()

In [ ]:
# Heatmap WITH oxygen
fig, ax = plt.subplots(figsize=(18, 16))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.5,
    cbar_kws={"shrink": 0.7},
    ax=ax,
    annot_kws={"size": 8},
)
ax.set_title(
    "Correlation Matrix — All Informative Elements (including Oxygen)",
    fontsize=14,
    pad=15,
)
plt.tight_layout()
plt.savefig(
    "outputs/NFI_relevant_elements_only__correlation_matrix_all_informative_elements_including_oxygen.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()

In [ ]:
# Heatmap WITHOUT oxygen
no_o_cols = [c for c in informative_cols if c != "o"]
corr_no_o = informative_df[no_o_cols].corr()

fig, ax = plt.subplots(figsize=(17, 15))
mask2 = np.triu(np.ones_like(corr_no_o, dtype=bool))
sns.heatmap(
    corr_no_o,
    mask=mask2,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.5,
    cbar_kws={"shrink": 0.7},
    ax=ax,
    annot_kws={"size": 8},
)
ax.set_title(
    "Correlation Matrix — Informative Elements (excluding Oxygen)", fontsize=14, pad=15
)
plt.tight_layout()
plt.savefig(
    "outputs/NFI_relevant_elements_only__correlation_matrix_informative_elements_excluding_oxygen.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()

In [ ]:
# Multivariate check for oxygen
X_all = nfi_df[informative_cols].drop(columns=["o"]).fillna(0)
y_o = nfi_df["o"].fillna(0)

reg = LinearRegression()
reg.fit(X_all, y_o)
print(f"Multivariate R² (all elements → O): {reg.score(X_all, y_o):.4f}")

Multivariate R² (all elements → O): 0.9494


In [35]:
# Sanity check ... Multivariate for Pb
X_all = nfi_df[informative_cols].drop(columns=["pb"]).fillna(0)
y_o = nfi_df["pb"].fillna(0)

reg = LinearRegression()
reg.fit(X_all, y_o)
print(f"Multivariate R² (all elements → Pb): {reg.score(X_all, y_o):.4f}")

Multivariate R² (all elements → Pb): 0.9337


So ... oxygen is not unique in this multivariate sense ...

**"Closed Composition" problem (Aitchison, 1986)**
... "the sum constraint on compositional data creates spurious correlations and multicollinearity, which can lead to misleading interpretations of relationships between elements."

In [31]:
# Deoxygenated share of each element as % of grand total sum across all observations
element_sums = deoxygenated_df[no_o_cols].sum()
grand_total = element_sums.sum()
element_pct = (element_sums / grand_total * 100).sort_values(ascending=False)

print("Top 10 elements by share of total elemental composition (all observations):\n")
display(element_pct.head(10).rename("% of grand total").to_frame().round(2))

Top 10 elements by share of total elemental composition (all observations):



,% of grand total
ba,27.59
pb,12.48
fe,12.35
cu,10.23
s,8.22
zn,5.67
sb,5.60
si,4.43
al,3.96
ca,2.21


**Conclusion:**
   - Keep oxygen as a feature. Even weak negative signal is informative for
     a tree-based model (XGBoost can exploit it).
   - Removing O risks throwing away a weak but real signal; keeping it adds
     at most marginal noise for a tree model that can learn its low importance.
   - Consider circumstances where oxygen may be diluting signal.
     Compare/contrast how those features are different when oxygen is removed.